In [105]:
# Set up sys.path so that 'src/spindle_dev' is importable as 'spindle_dev'
import sys
import time
import importlib  
from pathlib import Path

import numpy as np
import scanpy as sc
from tqdm.auto import tqdm

project_root = '..'
src_path = Path(project_root) / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import spindle_dev
# Reload to pick up code changes without restarting the kernel
importlib.reload(spindle_dev)

import spindle_dev.metrics as metrics
import spindle_dev.index as index
import spindle_dev.preprocessing as preprocessing
import spindle_dev.plotting as plotting
import spindle_dev.typing as typing
import spindle_dev.test as test
import spindle_dev.search as search

In [106]:
def prepare_to_index(adata):
    """
    Prepare standard data object for indexing.
    """
    coords = adata.obsm["spatial"]
    tiles = preprocessing.build_quadtree_tiles(coords, max_pts=200, min_side=0.0, max_depth=40)
    num_genes = adata.n_vars
    genes_work, gene_idx = spindle_dev.preprocessing.topvar_genes(adata, G=num_genes)  
    tile_covs = spindle_dev.preprocessing.build_tile_covs_full(adata, tiles, gene_idx, n_jobs=8, eps=1e-6)

    return tiles, tile_covs, genes_work


def run_index(tiles, tile_covs, genes_work, adata, resolution=0.2, min_final_size=20):
    """
    Run indexing workflow.
    """
    data = index.ProcessedData(tiles, tile_covs, genes_work, adata.n_obs)
    data.reduce_dim(num_pca_components=30, n_components=2, do_umap=True)
    data.cluster_spds(cluster_distance="tree", cluster_method="leiden", resolution=resolution)
    data.assign_label_to_spots()
    data.get_corr_mean_by_cluster()
    out_dict = data.get_adaptive_runs(find_blocks=True, with_size_guard=True, min_final_size=min_final_size, max_final_size=100)
    return data, out_dict

In [107]:
def load_and_split_data(adata_path, test_ratio=0.005, seed=4):
    print(f"Reading data from {adata_path}...")
    adata = sc.read_h5ad(adata_path)
    adata = adata[adata.obs.loc[adata.obs.Cluster != "Unlabeled"].index, :].copy()

    print("Preparing data for indexing...")
    tiles, tile_covs, genes_work = prepare_to_index(adata)

    np.random.seed(seed) 
    num_total_tiles = len(tiles)
    num_test = int(num_total_tiles * test_ratio)

    all_indices = np.arange(num_total_tiles)
    test_idx = np.random.choice(all_indices, size=num_test, replace=False)
    train_idx = np.setdiff1d(all_indices, test_idx)

    train_tiles = [tiles[i] for i in train_idx]
    train_tile_covs = [tile_covs[i] for i in train_idx]
    test_tiles = [tiles[i] for i in test_idx]
    test_tile_covs = [tile_covs[i] for i in test_idx]

    print(f"Total tiles: {num_total_tiles} | Training/Indexed: {len(train_tiles)} | Held out/Testing: {len(test_tiles)}")

    return adata, genes_work, train_tiles, train_tile_covs, test_tiles, test_tile_covs, train_idx, test_idx

In [108]:
def configure_and_build_dag(data):
    print("Configuring adaptive epsilons for blocks...")
    epsilon_block_wise_dict = {}
    epsilon_dict = {}
    for cluster_id in set(data.labels):
        eps_per_block, eps_elbow_per_block, eps = index.choose_adaptive_epsilons(data, cluster_id, k_target_per_block=64)
        epsilon_block_wise_dict[int(cluster_id)] = eps_elbow_per_block
        epsilon_dict[int(cluster_id)] = eps

    config = typing.IndexConfig()
    config.epsilon_dict = epsilon_dict
    config.epsilon_block_wise_dict = epsilon_block_wise_dict
    config.threshold_type = 'block_wise'
    config.kmean_method = 'epsilon_net'

    print("Creating index DAG...")
    dag_dict, stat, dist_list = index.index_spds(data, config=config)
    
    return dag_dict, config


def extract_query_matrices(test_tile_covs):
    query_matrices = []
    for q_dict in test_tile_covs:
        if isinstance(q_dict, dict):
            query_matrices.append(q_dict.get('cov', q_dict.get('matrix', q_dict)))
        else:
            query_matrices.append(q_dict)
    return query_matrices


In [109]:
def perform_search(query_matrices, data, dag_dict, config, budget_multiplier=0.7):
    search_cfg = search.SearchConfig(max_results=20, debug=False, max_failed_starts=10, max_failed_paths=50, total_paths_limit=100)

    print(f"Starting blind holdout validation for {len(query_matrices)} unseen queries...")
    print("-" * 65)

    print("Step 1/2: Assigning queries to Covariance-Niches using latent space...")
    assign_start = time.time()
    predicted_clusters = search.assign_clusters_to_new_spds(query_matrices, data)
    print(f"Assignment complete in {time.time() - assign_start:.3f}s\n")

    print("Step 2/2: Performing distance-budgeted search across DAG...")
    search_start = time.time()
    all_matched_train_ids = []

    for j, cluster_id in enumerate(tqdm(predicted_clusters, desc="Querying Index", leave=True)):
        cluster_id = int(cluster_id)
        index_handle = dag_dict[cluster_id]
        epsilon = config.epsilon_dict[cluster_id]
        num_blocks = len(index_handle.sorted_blocks)
        
        # Budget computation
        budget = float(epsilon) * float(num_blocks) * float(budget_multiplier)

        q_spd = query_matrices[j]
        perm = data.perm_list[cluster_id]
        q_spd_perm = q_spd[np.ix_(perm, perm)]
        query_block_runs = data.block_dict[cluster_id]

        results = search.search_index(
            index_handle,
            q_spd_perm,
            [],
            query_block_runs,
            budget,
            config=search_cfg,
        )

        matched_ids_for_query = []
        if results.paths:
            for rank, path in enumerate(results.paths, start=1):
                member_sets = []
                for node_id in path.node_path:
                    node = index_handle.nodes[node_id]
                    members = getattr(getattr(node, "metadata", None), "members", [])
                    spd_ids = {int(spd_id) for spd_id, _ in members}
                    member_sets.append(spd_ids)

                intersect_ids = set.intersection(*member_sets) if member_sets else set()
                matched_ids_for_query.extend(sorted(intersect_ids))
            
        all_matched_train_ids.append(matched_ids_for_query)

    search_time = time.time() - search_start
    print("-" * 65)
    print(f"Index Querying Complete! Total time: {search_time:.3f}s ({search_time/len(predicted_clusters):.4f}s per query)")
    
    return predicted_clusters, all_matched_train_ids


def summarize_hits(all_matched_train_ids, predicted_clusters):
    print("\n" + "="*40)
    print("           QUERY HITS SUMMARY")
    print("="*40)
    for j, hits in enumerate(all_matched_train_ids):
        target_niche = int(predicted_clusters[j])
        print(f"Query {j:3d} [Niche {target_niche:2d}]: {len(hits):4d} matches found")
    print("="*40 + "\n")

In [110]:
def log_spd(M, eps=1e-6):
    M = 0.5 * (M + M.T)
    w, V = np.linalg.eigh(M)
    w = np.maximum(w, eps)
    return (V * np.log(w)) @ V.T

def get_ordinal(n):
    if 11 <= (n % 100) <= 13:
        return f"{n}th"
    suffixes = ["th", "st", "nd", "rd", "th", "th", "th", "th", "th", "th"]
    return str(n) + suffixes[n % 10]

def calc_corr(mat1, mat2, perm, block_runs):
    m1_perm = mat1[np.ix_(perm, perm)]
    m2_perm = mat2[np.ix_(perm, perm)]
    
    # Arrays to hold the concatenated block data
    v1_all = []
    v2_all = []
    
    for (start, end) in block_runs:
        b1 = m1_perm[start:end, start:end]
        b2 = m2_perm[start:end, start:end]
        
        # Get upper triangle excluding the diagonal
        iu = np.triu_indices(b1.shape[0], k=1) 
        v1_all.extend(b1[iu])
        v2_all.extend(b2[iu])
        
    v1_arr = np.array(v1_all)
    v2_arr = np.array(v2_all)
    
    if len(v1_arr) > 1 and np.std(v1_arr) > 0 and np.std(v2_arr) > 0:
        c = np.corrcoef(v1_arr, v2_arr)[0, 1]
        return c if not np.isnan(c) else 0.0
    return 0.0


In [111]:
def evaluate_block_diagonalized_correlation(test_tile_covs, train_tile_covs, train_idx, predicted_clusters, all_matched_train_ids, data , plot = False):
    print("\n" + "="*50)
    print("TASK 1: Block-Diagonalized Correlation Permutation Test")
    print("="*50)
    
    overall_p_values = []
    global_to_local_train_map = {global_id: local_idx for local_idx, global_id in enumerate(train_idx)}
    
    for i, q_dict in enumerate(test_tile_covs):
        q_spd = q_dict if not isinstance(q_dict, dict) else q_dict.get('cov', q_dict.get('matrix', q_dict))
        target_niche = int(predicted_clusters[i])
        
        perm = data.perm_list[target_niche]
        block_runs = data.block_dict[target_niche]

        if not all_matched_train_ids[i]:
            print(f"Query {i:3d}: Spindle found NOTHING (0 candidates).")
            continue

        true_match_corr = -float('inf')
        for match_global_idx in all_matched_train_ids[i]:
            local_match_idx = global_to_local_train_map.get(match_global_idx)
            if local_match_idx is None:
                continue

            t_spd = train_tile_covs[local_match_idx]
            t_spd = t_spd if not isinstance(t_spd, dict) else t_spd.get('cov')
            corr = calc_corr(q_spd, t_spd, perm, block_runs)
            if corr > true_match_corr:
                true_match_corr = corr

        if true_match_corr == -float('inf'):
            print(f"Query {i:3d}: Spindle found matches, but none were valid in train_idx.")
            continue
        
        random_indices = np.random.choice(len(train_tile_covs), size=500, replace=False)
        random_corrs = []
        for idx in random_indices:
            r_spd = train_tile_covs[idx]
            r_spd = r_spd if not isinstance(r_spd, dict) else r_spd.get('cov')
            random_corrs.append(calc_corr(q_spd, r_spd, perm, block_runs))
            
        avg_random_corr = np.mean(random_corrs)
        p_value = np.sum(np.array(random_corrs) >= true_match_corr) / 100.0
        overall_p_values.append(p_value)
        
        # Plot Background Distribution vs. True Match
        if plot:
            import matplotlib.pyplot as plt
            from pathlib import Path
            
            plot_dir = Path("correlation_plots")
            plot_dir.mkdir(exist_ok=True)
            
            plt.figure(figsize=(8, 5))
            plt.hist(random_corrs, bins=20, color='lightgray', edgecolor='black', alpha=0.7, label='Random Background (n=100)')
            plt.axvline(true_match_corr, color='red', linestyle='dashed', linewidth=2, label=f'Spindle Best Match ({true_match_corr:.2f})')
            plt.title(f'Query {i} (Niche {target_niche}): Block-Diagonalized Correlation\nEmpirical p-value = {p_value:.3f}')
            plt.xlabel('Pearson Correlation (Block Sum)')
            plt.ylabel('Frequency')
            plt.legend()
            plt.tight_layout()
            
            # Save plot to disk
            plt.savefig(plot_dir / f'query_{i:03d}_corr_distribution.png', dpi=150)
            plt.close()
        
        print(f"Query {i:3d}: True Match Corr = {true_match_corr:.3f}, Avg Random Corr = {avg_random_corr:.3f}, p-value = {p_value:.3f}")
    if overall_p_values:
        print(f"\nMean p-value across queries: {np.mean(overall_p_values):.3f}")



In [112]:
def evaluate_brute_force_approximation(test_tile_covs, train_tile_covs, train_idx, predicted_clusters, all_matched_train_ids, data):
    print("\n" + "="*60)
    print("TASK 2:Brute-Force Approximation Benchmark")
    print("="*60)
    
    global_to_local_train_map = {global_id: local_idx for local_idx, global_id in enumerate(train_idx)}
    
    for i, q_dict in enumerate(test_tile_covs):
        if not all_matched_train_ids[i]:
            continue
            
        target_niche = int(predicted_clusters[i])
        
        # 1. Get the architecture for this specific niche
        perm = data.perm_list[target_niche]
        block_runs = data.block_dict[target_niche]
        
        # 2. Extract, permute, and pre-compute logs for the query blocks
        q_spd = q_dict if not isinstance(q_dict, dict) else q_dict.get('cov', q_dict.get('matrix', q_dict))
        q_perm = q_spd[np.ix_(perm, perm)]
        
        q_blocks_log = []
        for (start, end) in block_runs:
            q_blocks_log.append(log_spd(q_perm[start:end, start:end]))
            
        # 3. Restrict brute force search to ONLY the training tiles in this niche
        # data.labels aligns perfectly with train_tile_covs
        niche_train_indices = [idx for idx, lab in enumerate(data.labels) if int(lab) == target_niche]
        
        distances = []
        
        # 4. Calculate Block-Wise distance for the niche
        for t_idx in niche_train_indices:
            t_spd = train_tile_covs[t_idx]
            t_spd = t_spd if not isinstance(t_spd, dict) else t_spd.get('cov')
            t_perm = t_spd[np.ix_(perm, perm)]
            
            total_block_dist = 0.0
            for b_idx, (start, end) in enumerate(block_runs):
                t_block = t_perm[start:end, start:end]
                t_block_log = log_spd(t_block)
                
                # Calculate Frobenius norm and normalize by block size (sqrt(p))
                diff = q_blocks_log[b_idx] - t_block_log
                p_block = t_block.shape[0]
                total_block_dist += np.linalg.norm(diff, ord='fro') / np.sqrt(p_block)
                
            distances.append((total_block_dist, t_idx))
            
        # Sort to find the true Block-Wise nearest neighbors in this Niche
        distances.sort(key=lambda x: x[0]) 
        
        # 5. Evaluate Spindle's BEST match from its search pool
        closest_dist = distances[0][0] if len(distances) > 0 else float('inf')
        
        if not all_matched_train_ids[i]:
            print(f"Query {i:3d} (Niche {target_niche}): Exact closest dist = {closest_dist:.3f} | Spindle found NOTHING.")
            continue

        spindle_best_dist = float('inf')
        spindle_best_rank = -1
        
        for match_global_idx in all_matched_train_ids[i]:
            local_match_idx = global_to_local_train_map.get(match_global_idx)
            if local_match_idx is None:
                continue
                
            match_rank = -1
            match_dist = -1
            for r, (d, idx) in enumerate(distances):
                if idx == local_match_idx:
                    match_rank = r + 1
                    match_dist = d
                    break
                    
            if match_dist != -1 and match_dist < spindle_best_dist:
                spindle_best_dist = match_dist
                spindle_best_rank = match_rank
                
        if spindle_best_rank == -1:
            print(f"Query {i:3d} (Niche {target_niche}): Exact closest dist = {closest_dist:.3f} | Spindle found NOTHING in this niche.")
        else:
            print(f"Query {i:3d} (Niche {target_niche}): Exact closest dist = {closest_dist:.3f}, Spindle dist = {spindle_best_dist:.3f} | Spindle found {get_ordinal(spindle_best_rank)} closest neighbor.")


In [113]:
adata_path = "../dataset/adata.h5ad"

In [114]:
# 1. Load and split data
adata, genes_work, train_tiles, train_tile_covs, test_tiles, test_tile_covs, train_idx, test_idx = load_and_split_data(adata_path)

Reading data from ../dataset/adata.h5ad...
Preparing data for indexing...
Total tiles: 2081 | Training/Indexed: 2071 | Held out/Testing: 10


In [115]:
# 2. Run index
print("Running index...")
data, out_dict = run_index(train_tiles, train_tile_covs, genes_work, adata, resolution=0.2, min_final_size=15)

[2026-04-03 23:03:38,727] INFO spindle_dev.index: Clustering SPD-s using 'tree' distance.
[2026-04-03 23:03:38,727] INFO spindle_dev.index: Building ultrametric features from SPD matrices.


Running index...


[2026-04-03 23:03:45,607] INFO spindle_dev.index: Computing latent features from the tree representations.
[2026-04-03 23:03:46,421] INFO spindle_dev.index: Reducing latent features to 30 dimensions using PCA.
[2026-04-03 23:03:51,806] INFO spindle_dev.index: Explained variance ratios by PCA components: [0.06883564 0.05389878 0.02647495 0.01915427 0.01302046 0.01024849
 0.00782635 0.00736611 0.00664102 0.00646657 0.00501463 0.00464886
 0.00424013 0.00416745 0.00382564 0.00365452 0.00356241 0.00350148
 0.00333444 0.00317451 0.00304889 0.00302828 0.00301846 0.00294038
 0.002912   0.00286834 0.00279509 0.00277205 0.00271221 0.00269902]
[2026-04-03 23:03:51,807] INFO spindle_dev.index: Reducing latent features to 2 dimensions using UMAP.
d:\SPINDLE\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[2026-04-03 23:03:59,304] INFO spindle_dev.index: Clustering SPD-s using 'tree' distance.
[2026

In [116]:
# 3. Configure and build DAG
dag_dict, config = configure_and_build_dag(data)

Configuring adaptive epsilons for blocks...


[2026-04-03 23:04:10,250] INFO spindle_dev.index: Processing cluster 0
[2026-04-03 23:04:10,251] INFO spindle_dev.index: Building SPD index with epsilon=7.603499228740438
[2026-04-03 23:04:10,251] INFO spindle_dev.index: Step 1: Cluster blocks within each class of SPD matrices.
[2026-04-03 23:04:10,346] INFO spindle_dev.index: Cluster 0: 737 SPDs, 16 blocks
[2026-04-03 23:04:10,347] INFO spindle_dev.index:  Using epsilon-net clustering for block 0


Creating index DAG...


[2026-04-03 23:04:10,427] INFO spindle_dev.index:  Finished block 0 in 0.08 seconds, found 2 clusters.
[2026-04-03 23:04:10,429] INFO spindle_dev.index:  Using epsilon-net clustering for block 1
[2026-04-03 23:04:10,526] INFO spindle_dev.index:  Finished block 1 in 0.10 seconds, found 7 clusters.
[2026-04-03 23:04:10,527] INFO spindle_dev.index:  Using epsilon-net clustering for block 2
[2026-04-03 23:04:10,601] INFO spindle_dev.index:  Finished block 2 in 0.07 seconds, found 4 clusters.
[2026-04-03 23:04:10,601] INFO spindle_dev.index:  Using epsilon-net clustering for block 3
[2026-04-03 23:04:10,671] INFO spindle_dev.index:  Finished block 3 in 0.07 seconds, found 4 clusters.
[2026-04-03 23:04:10,671] INFO spindle_dev.index:  Using epsilon-net clustering for block 4
[2026-04-03 23:04:10,739] INFO spindle_dev.index:  Finished block 4 in 0.07 seconds, found 3 clusters.
[2026-04-03 23:04:10,740] INFO spindle_dev.index:  Using epsilon-net clustering for block 5
[2026-04-03 23:04:10,802]

In [117]:
# 4. Extract queries
query_matrices = extract_query_matrices(test_tile_covs)

In [118]:
# 5. Perform search
predicted_clusters, all_matched_train_ids = perform_search(query_matrices, data, dag_dict, config)

# 6. Summarize hits
summarize_hits(all_matched_train_ids, predicted_clusters)

Starting blind holdout validation for 10 unseen queries...
-----------------------------------------------------------------
Step 1/2: Assigning queries to Covariance-Niches using latent space...
Assignment complete in 0.043s

Step 2/2: Performing distance-budgeted search across DAG...


Querying Index:   0%|          | 0/10 [00:00<?, ?it/s]

-----------------------------------------------------------------
Index Querying Complete! Total time: 0.577s (0.0577s per query)

           QUERY HITS SUMMARY
Query   0 [Niche  0]:   60 matches found
Query   1 [Niche  0]:   40 matches found
Query   2 [Niche  0]:   27 matches found
Query   3 [Niche  1]:  210 matches found
Query   4 [Niche  0]:   46 matches found
Query   5 [Niche  3]:   33 matches found
Query   6 [Niche  1]:   38 matches found
Query   7 [Niche  1]:  129 matches found
Query   8 [Niche  0]:   34 matches found
Query   9 [Niche  2]:   47 matches found



In [119]:
evaluate_block_diagonalized_correlation(test_tile_covs, train_tile_covs, train_idx, predicted_clusters, all_matched_train_ids, data)



TASK 1: Block-Diagonalized Correlation Permutation Test
Query   0: True Match Corr = 0.954, Avg Random Corr = 0.614, p-value = 0.040
Query   1: True Match Corr = 0.945, Avg Random Corr = 0.630, p-value = 0.000
Query   2: True Match Corr = 0.951, Avg Random Corr = 0.555, p-value = 0.090
Query   3: True Match Corr = 0.932, Avg Random Corr = 0.323, p-value = 0.060
Query   4: True Match Corr = 0.968, Avg Random Corr = 0.573, p-value = 0.010
Query   5: True Match Corr = 0.941, Avg Random Corr = 0.556, p-value = 0.030
Query   6: True Match Corr = 0.908, Avg Random Corr = 0.473, p-value = 0.030
Query   7: True Match Corr = 0.911, Avg Random Corr = 0.473, p-value = 0.010
Query   8: True Match Corr = 0.955, Avg Random Corr = 0.596, p-value = 0.100
Query   9: True Match Corr = 0.931, Avg Random Corr = 0.503, p-value = 0.010

Mean p-value across queries: 0.038


In [120]:
evaluate_brute_force_approximation(test_tile_covs, train_tile_covs, train_idx, predicted_clusters, all_matched_train_ids, data)



TASK 2:Brute-Force Approximation Benchmark
Query   0 (Niche 0): Exact closest dist = 58.618, Spindle dist = 61.171 | Spindle found 13th closest neighbor.
Query   1 (Niche 0): Exact closest dist = 41.904, Spindle dist = 43.821 | Spindle found 9th closest neighbor.
Query   2 (Niche 0): Exact closest dist = 49.947, Spindle dist = 52.324 | Spindle found 2nd closest neighbor.
Query   3 (Niche 1): Exact closest dist = 28.675, Spindle dist = 32.754 | Spindle found 3rd closest neighbor.
Query   4 (Niche 0): Exact closest dist = 44.891, Spindle dist = 46.744 | Spindle found 3rd closest neighbor.
Query   5 (Niche 3): Exact closest dist = 48.370, Spindle dist = 48.370 | Spindle found 1st closest neighbor.
Query   6 (Niche 1): Exact closest dist = 72.229, Spindle dist = 75.274 | Spindle found 16th closest neighbor.
Query   7 (Niche 1): Exact closest dist = 33.683, Spindle dist = 36.415 | Spindle found 11th closest neighbor.
Query   8 (Niche 0): Exact closest dist = 47.978, Spindle dist = 51.016 |